# MindRise Exam-Readiness Model: Comparison & Evaluation Notebook

Loads the real JSON reports produced by the research-grade ML pipeline (`model_comparison.py`, `evaluate.py`, `explain.py`, `train_multioutput.py`, `bias_fairness_report.py`) and renders them as tables/charts. Every number here traces to a file under `models/` - nothing in this notebook is computed independently of the training scripts, so re-running training and re-running this notebook always stay consistent.

See `docs/ML_RESEARCH_METHODOLOGY.md` for the full narrative methodology this notebook's numbers back.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

MODELS = Path('models')

def load(name):
    path = MODELS / name
    if not path.exists():
        print(f'{name} not found yet - run the corresponding training script first.')
        return None
    return json.loads(path.read_text())

comparison = load('model_comparison_report.json') or load('metadata.json')
evaluation = load('evaluation_report.json')
explainability = load('explainability_report.json')
multioutput = load('multioutput_metadata.json')
bias = load('bias_fairness_report.json')
registry = load('registry.json')

## 1. Dataset composition

In [ ]:
if comparison:
    print('Training rows:', comparison.get('training_rows'))
    counts = comparison.get('data_source_counts', {})
    if counts:
        pd.Series(counts).plot(kind='bar', title='Hybrid dataset composition', ylabel='rows')
        plt.tight_layout()
        plt.show()

## 2. Model screening (9 candidates, 5-fold CV)

In [ ]:
if comparison and 'screening_round' in comparison:
    screening_df = pd.DataFrame({
        name: {'cv_f1_macro_mean': r['cv_f1_macro_mean'], 'cv_f1_macro_std': r['cv_f1_macro_std'], 'elapsed_seconds': r['elapsed_seconds']}
        for name, r in comparison['screening_round'].items()
    }).T.sort_values('cv_f1_macro_mean', ascending=False)
    display(screening_df)
    screening_df['cv_f1_macro_mean'].plot(kind='barh', xerr=screening_df['cv_f1_macro_std'], title='Screening macro-F1 by model (5-fold CV)')
    plt.tight_layout()
    plt.show()
else:
    print('Run model_comparison.py to populate this section.')

## 3. Hyperparameter optimization: default vs. optimized

In [ ]:
if comparison and 'default_vs_optimized_test_f1' in comparison:
    hpo_df = pd.DataFrame(comparison['default_vs_optimized_test_f1']).T
    hpo_df['improvement'] = hpo_df['optimized_test_f1_macro'] - hpo_df['default_test_f1_macro']
    display(hpo_df)
    print(f"\nSelected model: {comparison.get('best_model')}")
    print(f"\nTabNet exclusion rationale:\n{comparison.get('tabnet_exclusion_rationale', '(see ML_RESEARCH_METHODOLOGY.md Sec 5.2)')}")

## 4. Comprehensive evaluation

In [ ]:
if evaluation:
    display(pd.Series(evaluation['core_metrics'], name='value').to_frame())
    print('\nOverfitting diagnosis:', evaluation['overfitting_diagnosis']['diagnosis'])
    print('10-fold CV macro-F1:', evaluation['cross_validation']['ten_fold_cv']['mean'])
    print('Repeated 5x3 CV macro-F1:', evaluation['cross_validation']['repeated_cv_5x3']['mean'])
    ci = evaluation['bootstrap_confidence_intervals']['f1_macro']
    print(f"Bootstrap 95% CI on macro-F1: [{ci['ci_lower']:.4f}, {ci['ci_upper']:.4f}]")
else:
    print('Run evaluate.py to populate this section.')

In [ ]:
if evaluation:
    ts, tr, va = evaluation['overfitting_diagnosis']['train_sizes'], evaluation['overfitting_diagnosis']['train_scores_mean'], evaluation['overfitting_diagnosis']['val_scores_mean']
    plt.plot(ts, tr, label='train')
    plt.plot(ts, va, label='cross-validation')
    plt.xlabel('training set size'); plt.ylabel('macro-F1'); plt.title('Learning curve'); plt.legend()
    plt.show()

## 5. Per-data-source performance (real vs. synthetic generalization check)

In [ ]:
if evaluation and evaluation.get('per_data_source_performance'):
    display(pd.DataFrame(evaluation['per_data_source_performance']).T)

## 6. Explainability: global SHAP importance

In [ ]:
if explainability:
    imp_df = pd.DataFrame(explainability['global_shap_importance'], columns=['feature', 'mean_abs_shap']).set_index('feature')
    display(imp_df.head(15))
    imp_df.head(15).sort_values('mean_abs_shap').plot(kind='barh', title='Top 15 features by mean |SHAP|', legend=False)
    plt.tight_layout()
    plt.show()
    lime = explainability.get('lime_cross_check', {})
    print('LIME/SHAP mean top-5 feature overlap:', lime.get('mean_top5_feature_overlap_with_shap'))
else:
    print('Run explain.py to populate this section.')

## 7. Multi-output models (real OULAD ground truth)

In [ ]:
if multioutput:
    display(pd.DataFrame(multioutput['targets']).T)
    print('\nExcluded outputs and why:')
    for name, reason in multioutput.get('excluded_outputs', {}).items():
        print(f'  {name}: {reason}')
else:
    print('Run train_multioutput.py to populate this section.')

## 8. Bias / fairness analysis (real OULAD demographics)

In [ ]:
if bias:
    for group, values in bias['outcome_distribution'].items():
        print(f'\n{group}:')
        for k, v in values.items():
            print(f"  {k}: n={v['n']}, ready_share={v['label_distribution'].get('ready', 0):.3f}")
else:
    print('Run data_pipeline/bias_fairness_report.py to populate this section.')

## 9. Model version registry

In [ ]:
if registry and registry.get('versions'):
    display(pd.DataFrame(registry['versions']))
    print('\nLive version:', registry.get('live_version'))
else:
    print('No versions registered yet - run model_registry.py or retrain.py.')